In [ ]:
%pip install jsonschema
%pip install langchain
%pip install langchain_mistralai
%pip install pandas
%pip install sqlalchemy
%pip install sqlalchemy
%pip install langchain-community

We will create an **agent** that can **answer questions based on data** found in a **SQL Database**.
**LangChain** will be used as the **agent framework**.

For the agent, we need:
- A **system prompt**.
- The **tool(s)** the agent will use.

Let’s start with the **tool**.

We use SQLite as a SQL database. If the database does not already exist, it is created from the SQL script `employee.sql` in the `data` folder.

Additionally, we define the query function `query_sqlite3_db`.

In [24]:
import sqlite3
import os

sqlite_db="./data/employee.db"
sqlite_db_create="./data/employee.sql"
def init_sqlite3_db():
    try:
        if not os.path.exists(sqlite_db):
            connection = sqlite3.connect(sqlite_db)
            with open(sqlite_db_create, "r") as infile:
              content = infile.read() # Reads entire remaining content as a string
            #print(content)
            connection.executescript(content)
            connection.commit()
            connection.close()
        
    except Exception as e:
        print(f"Error running SQL query: {e}")

def query_sqlite3_db(query: str) -> list[any]:
    conn = sqlite3.connect(sqlite_db)
    c = conn.cursor()
    c.execute(query)
    matches = c.fetchall()
    c.close()
    return matches
    
init_sqlite3_db();


Let's test the sqlite3 query function:

In [25]:
query_sqlite3_db("SELECT count(*) FROM employees;")

[(201,)]

The answer is correct. We can now define the corresponding tool function.

In [31]:
from langchain.tools import tool
import sqlite3
@tool
def query_sql(query: str) -> list[any]:
    """Run a SQL query on the provided data."""
    try:
        return  query_sqlite3_db(query)
    except Exception as e:  
        serr=f"Error running SQL query: {e}"
        return [serr]  

Next, we create the **system prompt** to guide the agent. This prompt includes:
- Instructions on how to use the tool.
- A **schema description** of the data, so the agent can understand the structure of the data it is working with.

To generate the **schema description**, we use the `SQLDatabase` class from the LangChain Community contributions. This class allows us to extract the schema directly from the database, producing **CREATE TABLE statements** along with **comments** that include sample data for each table.

In [22]:
from sqlalchemy import ForeignKey
from sqlalchemy import String, Integer, Boolean, Date
from sqlalchemy.orm import DeclarativeBase
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship
from sqlalchemy import create_engine, MetaData, Table, Column
from sqlalchemy import inspect
from langchain_community.utilities.sql_database import SQLDatabase
engine = create_engine("sqlite:///data/employee.db", echo=False)
inspector = inspect(engine)
table_names=inspector.get_table_names()
db = SQLDatabase(engine)
sql_schema=db.get_table_info_no_throw([t.strip() for t in table_names])


In [27]:
print(sql_schema)


CREATE TABLE "assignedTo" (
	id VARCHAR NOT NULL, 
	name VARCHAR, 
	tasks_id VARCHAR NOT NULL, 
	PRIMARY KEY (id), 
	FOREIGN KEY(tasks_id) REFERENCES tasks (id)
)

/*
3 rows from assignedTo table:
id	name	tasks_id
E00001	Shannon Curtis	T732
E00002	Tommy Parker	T1855
E00003	Benjamin Archer	T224
*/


CREATE TABLE address (
	id VARCHAR NOT NULL, 
	street VARCHAR, 
	city VARCHAR, 
	contact_id VARCHAR NOT NULL, 
	PRIMARY KEY (id), 
	FOREIGN KEY(contact_id) REFERENCES contact (id)
)

/*
3 rows from address table:
id	street	city	contact_id
109e633c-090e-4851-853d-3d4569eb37b0	0492 Joe Grove Suite 306	West Michaelview	5c699cd7-7026-454d-bc58-bfd2bc77d767
333f108e-3b83-4e12-9597-7459fb5d5d6d	1366 Garcia Corners Suite 883	Fostermouth	08f7b02d-191b-4041-bbbd-92acb608d7c1
ef57d47b-d08c-45ac-a3c1-09b31efcf3fb	461 Paul Landing Apt. 055	West Brittany	37992716-c807-4aca-bfcc-8f90bfd4ac13
*/


CREATE TABLE certifications (
	id VARCHAR NOT NULL, 
	experience_id VARCHAR NOT NULL, 
	PRIMARY KEY (id), 
	F

Using the schema, we can create the prompt for the agent. Please note that the prompt is primarily based on the example from the **[LangChain documentation](https://docs.langchain.com/oss/python/langchain/sql-agent)**.

In [28]:
from langchain.agents import create_agent
from langchain_mistralai.chat_models import ChatMistralAI
import os
import io

template = f"""
Create a SQL query to get the required information from the data.

Here is the schema for the SQL database you will work with:
{sql_schema}

Given an input question, create a syntactically correct SQLite query to run, then look at the results of the query and return the answer.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.
You have access to tools for interacting with the database.
Only use the below tools. Only use the information returned by the below tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.
Only use tools if you are sure a query will get a correct answer. 

tools:
query_sql: returns the entities using a given SQL query.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
Always use double quotes for column names.

Always use the available tools to fetch data.
Only use the results from tools defined above to answer the question.
DO NOT use sample data shown in the table definitions!
DO NOT restrict the number of results with LIMIT clause!
"""

Now that we have the **tool function** and the **system prompt** for the agent, we can create the agent.

We will use the **Mistral AI model**: `mistral-large-latest` for the agent. Please note that I have saved the **API key** from Mistral in the environment variable `MISTRAL_API_KEY`.

In [33]:
llmm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0,
    max_retries=2
)


agent = create_agent(
    model=llmm,
    tools=[query_sql],
    system_prompt=template,
)

That’s it! Now we can ask the agent questions about the data, and it will generate the appropriate **SQL queries** to fetch the answers. If a question cannot be answered with the available data, the agent will respond accordingly.

In [ ]:
from langchain.messages import HumanMessage 
result = agent.invoke(
    {"messages": [HumanMessage("Hown many employees are there?")]}
)
result['messages'][-1].content

'There are **201 employees** in the database.'

The result is correct—there are **201 employees** in the dataset.
The query generated by the LLM is:

In [35]:
result['messages'][1].additional_kwargs["tool_calls"][0]

{'id': 'SsW5wtfVf',
 'function': {'name': 'query_sql',
  'arguments': '{"query": "SELECT COUNT(*) AS total_employees FROM employees;"}'},
 'index': 0}

The query created by the LLM is identical to the one we used earlier.
Now, let’s see how it performs with a more complex query.

In [37]:
from langchain.messages import HumanMessage 
result = agent.invoke(
    {"messages": [HumanMessage("Which employees have the greatest experience?")]}
)
print(result['messages'][-1].content)

The employees with the **greatest experience (10 years)** are:

1. Eileen Fitzgerald
2. Ashley Arroyo
3. Blake Mitchell
4. Stephanie Lewis
5. Chris Thomas
6. Brenda Wang
7. David Smith
8. Patrick Lee
9. Jim Werner
10. Abigail Briggs
11. Cathy Oconnor
12. Ricky Perry
13. Billy Bradley DVM
14. Matthew Caldwell
15. Robert Kelley
16. Alexander Guerrero IV
17. Rebecca King
18. Barbara Andrews
19. Dustin Newton
20. Sara Rivera
21. Duane Aguilar
22. April Cline
23. Robert Johnson
24. Willie Jones
25. Kelly Cardenas
26. Jennifer Anderson
27. Mary Fox
28. Albert Gonzalez
29. Timothy Mullins


The result is correct: **all employees have 10 years of experience**.

Let's look into the generated SQL query:

In [38]:
import json
args=json.loads(result['messages'][1].additional_kwargs["tool_calls"][0]['function']['arguments'])
print(args['query'])

SELECT e.name, ex.years AS experience_years
FROM employees e
JOIN profile p ON e.id = p.employees_id
JOIN contact c ON p.id = c.profile_id
JOIN skills s ON e.id = s.assignedTo_id
JOIN experience ex ON s.id = ex.skills_id
ORDER BY ex.years DESC


That’s interesting: the **SQL query** retrieves the experience of all employees. Indeed, the tool message containing the query results includes data for every employee.

In [39]:
result['messages'][2]

ToolMessage(content='[["Eileen Fitzgerald", 10], ["Ashley Arroyo", 10], ["Blake Mitchell", 10], ["Stephanie Lewis", 10], ["Chris Thomas", 10], ["Brenda Wang", 10], ["David Smith", 10], ["Patrick Lee", 10], ["Jim Werner", 10], ["Abigail Briggs", 10], ["Cathy Oconnor", 10], ["Ricky Perry", 10], ["Billy Bradley DVM", 10], ["Matthew Caldwell", 10], ["Robert Kelley", 10], ["Alexander Guerrero IV", 10], ["Rebecca King", 10], ["Barbara Andrews", 10], ["Dustin Newton", 10], ["Sara Rivera", 10], ["Duane Aguilar", 10], ["April Cline", 10], ["Robert Johnson", 10], ["Willie Jones", 10], ["Kelly Cardenas", 10], ["Jennifer Anderson", 10], ["Mary Fox", 10], ["Albert Gonzalez", 10], ["Timothy Mullins", 10], ["Amy Wilson", 9], ["Carolyn Cruz", 9], ["Christopher Thompson", 9], ["Anthony Jefferson", 9], ["Cheryl Lee", 9], ["Joanna Myers", 9], ["Jeffrey Johnson", 9], ["Miss Connie Levy", 9], ["Christopher Evans", 9], ["Carol Briggs", 9], ["Daniel Mueller", 9], ["Dr. William Berger", 9], ["Andrew House", 9

This is an interesting observation. On one hand, the query does not return the results we would expect; on the other hand, the LLM uses the query results to formulate a correct final answer.

This approach could lead to potential issues:

- **Using the fetched data**—for example, in diagrams or when exporting to a spreadsheet.
- **Handling large datasets**, as all the data would be loaded into the agent’s context.